In [ ]:
# === CHALLENGE 2 (TNF-alpha) : PRODUCTION SETUP ===
# Builds the CROPPED target. The smoke test measured 43.3 s per iteration on the full
# two-chain construct (304 + 70 residues) - 7.2x Challenge 1's rate for a 1.7x length
# increase, so the cost scales nearer L^3.7 than L^2. At that rate the whole GPU budget
# buys about six designs, which is not worth running.
#
# Cropping to a 22 A sphere around the switch site keeps all 25 site residues, both
# switch anchors (K166 of chain A, R108 of chain B) and - measured against the intact
# trimer - introduces ZERO artificially exposed surface at the site, fewer than the
# uncropped two-chain construct, because cropping also removes residues that chain C
# had been covering.
import glob, json, os, shutil, subprocess, sys, time
t0 = time.time()
OFF = 76
RADIUS = 22.0


def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print((r.stdout + r.stderr)[-1000:])
        if check:
            raise SystemExit("setup failed")
    return r


print("python", sys.version.split()[0], flush=True)
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip(), flush=True)

os.makedirs("/kaggle/working/params", exist_ok=True)
src = glob.glob("/kaggle/input/**/params/params_model_*.npz", recursive=True)
if src:
    print("reusing %d AF2 param files from the attached input" % len(src), flush=True)
    for f in src:
        dst = "/kaggle/working/params/" + os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy(f, dst)
else:
    print("downloading AF2 params (~3.5 GB) ...", flush=True)
    sh("wget -q -O /kaggle/working/af2.tar "
       "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
    sh("tar -xf /kaggle/working/af2.tar -C /kaggle/working/params")
    sh("rm -f /kaggle/working/af2.tar")
print("AF2 params present:", len(glob.glob("/kaggle/working/params/*.npz")), "files", flush=True)

sh("pip install -q biopython")
import urllib.request
import numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import protein_letters_3to1

os.makedirs("/kaggle/working/targets", exist_ok=True)
raw = "/kaggle/working/1TNF.pdb"
if not os.path.exists(raw):
    open(raw, "wb").write(urllib.request.urlopen(
        "https://files.rcsb.org/download/1TNF.pdb", timeout=180).read())

P = PDBParser(QUIET=True)
mdl = P.get_structure("m", raw)[0]
ca = {}
for ch in mdl:
    for r in ch:
        if r.id[0] == " " and "CA" in r:
            ca[(ch.id, r.id[1])] = r["CA"].get_coord()
centre = (ca[("A", 166 - OFF)] + ca[("B", 108 - OFF)]) / 2
keep = {k for k, v in ca.items() if k[0] in ("A", "B")
        and float(np.linalg.norm(v - centre)) <= RADIUS}


class Crop(Select):
    def accept_chain(self, c): return c.id in ("A", "B")
    def accept_residue(self, r): return r.id[0] == " " and (r.parent.id, r.id[1]) in keep
    def accept_atom(self, a): return a.element != "H"


TARGET = "/kaggle/working/targets/TNF_site_crop.pdb"
st = P.get_structure("t", raw)
io = PDBIO(); io.set_structure(st); io.save(TARGET, Crop())

# The map from construct order to the original chain and UniProt number. AfDesign
# CONCATENATES the target chains into one chain on output and renumbers from 1 - the
# smoke test showed chains {'A': 608, 'B': 140} where three chains were expected - so
# position in this list IS the output residue number minus one. Everything downstream
# maps through this rather than assuming a numbering.
order = []
for ch in P.get_structure("c", TARGET)[0]:
    for r in ch:
        if r.id[0] == " ":
            order.append(dict(chain=ch.id, pdb=r.id[1], uniprot=r.id[1] + OFF,
                              aa=protein_letters_3to1.get(r.get_resname(), "X")))
json.dump(dict(radius=RADIUS, n=len(order), order=order),
          open("/kaggle/working/construct_map.json", "w"))

SITE = [95, 105, 106, 107, 108, 109, 110, 111, 112, 155, 159, 161, 162, 163, 164, 165,
        166, 167, 168, 201, 203, 207, 220, 221, 223]
have = {(o["chain"], o["uniprot"]) for o in order}
print("cropped target: %d residues (A %d, B %d)"
      % (len(order), sum(1 for o in order if o["chain"] == "A"),
         sum(1 for o in order if o["chain"] == "B")), flush=True)
print("site residues retained: %d of %d"
      % (len({o["uniprot"] for o in order} & set(SITE)), len(SITE)), flush=True)
assert ("A", 166) in have and ("B", 108) in have, "a switch anchor was cropped away"
assert len({o["uniprot"] for o in order} & set(SITE)) == len(SITE), "site residue lost"
print("switch anchors present: K166(A), R108(B)", flush=True)

HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"

PROBE = ("python -c \"import jax, jax.lib; assert hasattr(jax.lib,'xla_bridge'); "
         "print(jax.__version__)\"")
r = subprocess.run(PROBE, shell=True, capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip().startswith("0.4.38"):
    print("installing jax 0.4.38 ...", flush=True)
    sh("pip install -q 'jax[cuda12]==0.4.38' 'jaxlib==0.4.38'")
sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")

chk = subprocess.run(
    "cd /kaggle/working && python -c \""
    "import jax; from colabdesign import mk_afdesign_model; "
    "m = mk_afdesign_model(protocol='binder', data_dir='/kaggle/working'); "
    "m.prep_inputs(pdb_filename='%s', chain='A,B', binder_len=65, hotspot='%s'); "
    "print('READY', jax.__version__, jax.devices()); "
    "print('target length seen by the model:', m._target_len)\"" % (TARGET, HOTSPOT),
    shell=True, capture_output=True, text=True)
print(chk.stdout[-500:])
if chk.returncode != 0:
    print("PREP FAILED:"); print(chk.stderr[-1500:])
print("TNF PRODUCTION SETUP DONE in %.1f min" % ((time.time() - t0) / 60))


In [ ]:
# === CHALLENGE 2 (TNF-alpha) : GENERATION ===
# Same shape as the EGFR round that worked: two workers, one per T4, a wall-clock guard
# so the batch always completes and saves, results appended to JSONL as each design
# finishes, and every design scored by an INDEPENDENT re-prediction of the discrete
# sequence at 3 recycles rather than by the optimiser's own number.
#
# Added here, because EGFR had to learn it the hard way: each design also reports which
# target residues it actually contacts, in UniProt numbering, mapped through the
# construct map. A confident interface is not necessarily the intended one, and finding
# that out at the end cost a round last time.
import os, subprocess, sys, textwrap, time

TIME_BUDGET_MIN = 330          # 5.5 h, leaving the rest of the weekly quota for switch work
LEN_MIN, LEN_MAX = 55, 80
SOFT, TEMP, HARD = 90, 40, 20
TARGET = "/kaggle/working/targets/TNF_site_crop.pdb"
HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"

WORKER = r'''
import json, os, random, sys, time
GPU, SHARD, BUDGET = sys.argv[1], int(sys.argv[2]), float(sys.argv[3])
os.environ["CUDA_VISIBLE_DEVICES"] = GPU          # must precede the jax import
SOFT, TEMP, HARD = int(sys.argv[4]), int(sys.argv[5]), int(sys.argv[6])
LEN_MIN, LEN_MAX = int(sys.argv[7]), int(sys.argv[8])
TARGET, HOTSPOT = sys.argv[9], sys.argv[10]

t0 = time.time()
def log(m): print("[w%d] %s [%.1f min]" % (SHARD, m, (time.time()-t0)/60), flush=True)

import numpy as np
from colabdesign import mk_afdesign_model, clear_mem

CMAP = json.load(open("/kaggle/working/construct_map.json"))["order"]
SITE = {95,105,106,107,108,109,110,111,112,155,159,161,162,163,164,165,166,167,168,
        201,203,207,220,221,223}

os.makedirs("/kaggle/working/tnf_out", exist_ok=True)
rec_path = "/kaggle/working/tnf_out/worker%d.jsonl" % SHARD
rng = random.Random(2000 + SHARD)

def epitope_of(pdb_path):
    """Target residues within 5 A of the binder, in UniProt numbering.

    AfDesign concatenates the target chains into one chain and renumbers from 1, so
    output residue n is CMAP[n-1]. Mapping through the map rather than assuming the
    numbering is the whole point.
    """
    from Bio.PDB import PDBParser
    from collections import Counter
    mdl = PDBParser(QUIET=True).get_structure("c", pdb_path)[0]
    sizes = Counter({ch.id: sum(1 for r in ch if r.id[0]==" ") for ch in mdl})
    tgt = max(sizes, key=lambda k: sizes[k]); bnd = min(sizes, key=lambda k: sizes[k])
    tres = [r for r in mdl[tgt] if r.id[0]==" "]
    # the file carries two models, so the chain holds each residue twice
    n = len(CMAP)
    if len(tres) >= 2*n: tres = tres[:n]
    if len(tres) != n:
        return None, "target chain %d residues, map has %d" % (len(tres), n)
    ba = np.array([a.get_coord() for r in mdl[bnd] if r.id[0]==" " for a in r])
    hits = []
    for i, r in enumerate(tres):
        ra = np.array([a.get_coord() for a in r])
        if np.min(np.linalg.norm(ra[:,None,:]-ba[None,:,:], axis=-1)) <= 5.0:
            hits.append(CMAP[i]["uniprot"])
    return sorted(set(hits)), None

printed = False
i = 0
while (time.time() - t0) / 60 < BUDGET:
    i += 1
    name = "tnf_w%d_%03d" % (SHARD, i)
    L = rng.randint(LEN_MIN, LEN_MAX)
    try:
        clear_mem()
        model = mk_afdesign_model(protocol="binder", data_dir="/kaggle/working")
        model.prep_inputs(pdb_filename=TARGET, chain="A,B", binder_len=L, hotspot=HOTSPOT)
        if not printed:
            log("target length %d, loss weights %s"
                % (model._target_len, dict(model.opt.get("weights", {}))))
            printed = True

        model.design_3stage(soft_iters=SOFT, temp_iters=TEMP, hard_iters=HARD)
        seq = model.get_seqs()[0]
        lg = model.aux["log"]
        design_iptm = float(lg.get("i_ptm", lg.get("ptm", 0.0)))

        try:
            model.predict(seq=seq, num_recycles=3, verbose=False)
        except TypeError:
            model.predict(seq=seq, verbose=False)
        lg = model.aux["log"]
        true_iptm = float(lg.get("i_ptm", lg.get("ptm", 0.0)))
        plddt = float(lg.get("plddt", 0.0))

        out = "/kaggle/working/tnf_out/%s.pdb" % name
        model.save_pdb(out)
        epi, err = epitope_of(out)
        on_site = sorted(set(epi) & SITE) if epi else []

        with open(rec_path, "a") as fh:
            fh.write(json.dumps(dict(name=name, length=L, sequence=seq,
                                     design_iptm=round(design_iptm,3),
                                     true_iptm=round(true_iptm,3),
                                     plddt=round(plddt,3),
                                     epitope=epi, on_site=on_site,
                                     epitope_error=err)) + "\n")
        log("%s len=%d design_iptm=%.3f TRUE_iptm=%.3f plddt=%.3f  epitope %s, %d on site %s"
            % (name, L, design_iptm, true_iptm, plddt,
               ("%d res" % len(epi)) if epi else ("ERR " + str(err)),
               len(on_site), on_site[:8]))
    except Exception as e:
        log("%s FAILED %s %s" % (name, type(e).__name__, str(e)[:200]))
log("worker done after %d trajectories" % i)
'''

os.makedirs("/kaggle/working/tnf_out", exist_ok=True)
with open("/kaggle/working/tnf_worker.py", "w") as fh:
    fh.write(textwrap.dedent(WORKER))

args = [str(SOFT), str(TEMP), str(HARD), str(LEN_MIN), str(LEN_MAX), TARGET, HOTSPOT]
procs = []
for shard, gpu in enumerate(["0", "1"]):
    p = subprocess.Popen([sys.executable, "-u", "/kaggle/working/tnf_worker.py",
                          gpu, str(shard), str(TIME_BUDGET_MIN)] + args,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    procs.append(p)
    print("launched worker %d on GPU %s" % (shard, gpu), flush=True)

import threading
def pump(p):
    for line in p.stdout:
        print(line.rstrip(), flush=True)
threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
[t.start() for t in threads]
for p in procs:
    p.wait()
[t.join(timeout=10) for t in threads]

import glob, json
recs = []
for f in sorted(glob.glob("/kaggle/working/tnf_out/worker*.jsonl")):
    for line in open(f):
        line = line.strip()
        if line:
            recs.append(json.loads(line))
recs.sort(key=lambda r: -r["true_iptm"])
json.dump(recs, open("/kaggle/working/tnf_round1.json", "w"), indent=1)

print("\n==== TNF GENERATION: %d designs ====" % len(recs))
print("%-13s %4s %7s %8s %7s %7s" % ("name","len","i_ptm","(design)","pLDDT","on site"))
for r in recs[:30]:
    print("%-13s %4d %7.3f %8.3f %7.3f %7d"
          % (r["name"], r["length"], r["true_iptm"], r["design_iptm"],
             r["plddt"], len(r["on_site"])))
good = [r for r in recs if r["true_iptm"] >= 0.50]
ok = [r for r in recs if r["true_iptm"] >= 0.35]
onsite = [r for r in recs if len(r["on_site"]) >= 5]
print("\n%d at i_ptm>=0.50, %d at >=0.35, best %.3f; %d contact 5+ site residues"
      % (len(good), len(ok), recs[0]["true_iptm"] if recs else 0.0, len(onsite)))
